# Spinelli Stage 5 — Parameter Sweep and Magnitude Test

This notebook tests whether the action-derived correction tensor found in Stage 4D remains useful beyond a single Alcubierre configuration.

Core tests:

1. **Universality**: does the fitted curvature coupling stay close to the action-derived value?

\[
eta_{
m action}=-1
\]

2. **Action closeness**: does the action-predicted tensor perform nearly as well as the best fitted HTR tensor?

\[
\epsilon_{
m action}/\epsilon_{
m fit} pprox 1
\]

3. **Magnitude**: does the action-derived correction have a non-negligible overlap with the classical Alcubierre negative-energy density?

\[
\mathcal{F}_{Q}=
rac{\int \max(Q^{
m action}_{00},0)\,dV}{\int |
ho_A|\,dV}
\]

For ordinary desktop runs, use `N_SWEEP=25`, `31`, or `41`. Use `N_CONFIRM=61` or `81` only for selected cases.


In [1]:
# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import gc
import json
import time
import zipfile
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import psutil
except Exception:
    psutil = None

OUTPUT_DIR = Path("stage5_parameter_sweep_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DPI = 170
DTYPE = np.float64

# Runtime mode.
# DIM=4 memory grows as N^4. A full 48-case sweep at N=81 is generally too heavy.
DIM = 4
N_SWEEP = 31

RUN_CONFIRMATION = True
N_CONFIRM = 61

EXTENT = 5.0
T_EXTENT = 0.4
DELTA_TAU = 0.04
INTERIOR_CROP = 3

# Start with this grid. Expand or reduce as needed.
V_VALUES = [0.25, 0.5, 0.75, 1.0]
SIGMA_VALUES = [0.5, 1.0, 2.0, 4.0]
R_VALUES = [2.0, 3.0, 4.0]

CONFIRMATION_CASES = [
    {"v_s": v_s, "sigma": sigma, "R": R}
    for v_s in [0.25, 0.5, 0.75, 1.0]
    for sigma in [2.0, 4.0]
    for R in [2.0, 3.0, 4.0]
]

BETA_ACTION = -1.0

print("Stage 5 configuration")
print(f"DIM={DIM}, N_SWEEP={N_SWEEP}, dtype={DTYPE}")
print(f"Parameter count = {len(V_VALUES) * len(SIGMA_VALUES) * len(R_VALUES)}")
if psutil:
    vm = psutil.virtual_memory()
    print(f"System RAM: {vm.total/1024**3:.1f} GiB, available: {vm.available/1024**3:.1f} GiB")


Stage 5 configuration
DIM=4, N_SWEEP=31, dtype=<class 'numpy.float64'>
Parameter count = 48
System RAM: 31.7 GiB, available: 14.9 GiB


In [2]:
# ============================================================
# 1. Utility and tensor functions
# ============================================================

def estimate_scalar_gib(N, dim, dtype=DTYPE):
    return (N ** dim) * np.dtype(dtype).itemsize / (1024 ** 3)


def memory_report(prefix=""):
    if psutil is None:
        return {}
    vm = psutil.virtual_memory()
    data = {
        "ram_total_gib": vm.total / 1024**3,
        "ram_available_gib": vm.available / 1024**3,
        "ram_used_percent": vm.percent,
    }
    if prefix:
        print(prefix, {k: round(v, 3) for k, v in data.items()})
    return data


def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()


def alcubierre_shape(rs, R, sigma):
    return (
        np.tanh(sigma * (rs + R)) - np.tanh(sigma * (rs - R))
    ) / (2.0 * np.tanh(sigma * R))


def get_interior_slices(field_ndim, crop):
    return tuple(slice(crop, -crop) for _ in range(field_ndim))


def l2_components(arr, crop=INTERIOR_CROP, tensor_rank=1):
    # L2 norm for component array with shape (dim,...), or (dim,dim,...).
    if tensor_rank == 0:
        interior = get_interior_slices(arr.ndim, crop)
        return float(np.sqrt(np.mean(arr[interior] ** 2)))

    component_shape = arr.shape[:tensor_rank]
    field_ndim = arr.ndim - tensor_rank
    interior = get_interior_slices(field_ndim, crop)

    total = 0.0
    if tensor_rank == 1:
        for a in range(component_shape[0]):
            total += np.mean(arr[a][interior] ** 2)
    elif tensor_rank == 2:
        for a in range(component_shape[0]):
            for b in range(component_shape[1]):
                total += np.mean(arr[a, b][interior] ** 2)
    else:
        raise ValueError("tensor_rank must be 0, 1, or 2")
    return float(np.sqrt(total))


def divergence_mixed_from_geometry(Tmix, Gamma, spacings):
    # Compute C_b = nabla_a T^a_b for a mixed tensor.
    dim = Tmix.shape[0]
    shape = Tmix.shape[2:]
    dtype = Tmix.dtype
    C = np.zeros((dim,) + shape, dtype=dtype)

    for b in range(dim):
        for a in range(dim):
            C[b] += np.gradient(Tmix[a, b], *spacings, edge_order=2)[a]
        for a in range(dim):
            for l in range(dim):
                C[b] += Gamma[a, a, l] * Tmix[l, b]
                C[b] -= Gamma[l, a, b] * Tmix[a, l]
    return C


def mix_tensor_up_down(Qcov, gi):
    dim = Qcov.shape[0]
    Qmix = np.zeros_like(Qcov)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                Qmix[a, b] += gi[a, c] * Qcov[c, b]
    return Qmix


def central_slice_scalar(field, geom):
    coords = geom["coords"]
    dim = geom["dim"]
    if dim == 3:
        t, x, y = coords
        it0 = int(np.argmin(np.abs(t)))
        return field[it0, :, :], x, y
    if dim == 4:
        t, x, y, z = coords
        it0 = int(np.argmin(np.abs(t)))
        iz0 = int(np.argmin(np.abs(z)))
        return field[it0, :, :, iz0], x, y
    raise ValueError("Only DIM=3 or DIM=4 supported")


def plot_central_scalar(field, geom, title, colorbar_label, filename):
    data, x, y = central_slice_scalar(field, geom)
    plt.figure(figsize=(7, 6))
    plt.imshow(data.T, extent=[x[0], x[-1], y[0], y[-1]], origin="lower", aspect="equal")
    plt.colorbar(label=colorbar_label)
    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    savefig(OUTPUT_DIR / filename)


In [3]:
# ============================================================
# 2. Geometry construction and Einstein tensor extraction
# ============================================================

def build_alcubierre_geometry(
    dim=4,
    N=31,
    R=3.0,
    sigma=1.0,
    v_s=0.5,
    extent=EXTENT,
    t_extent=T_EXTENT,
    delta_tau=DELTA_TAU,
    dtype=DTYPE,
):
    if dim not in (3, 4):
        raise ValueError("dim must be 3 or 4")

    coords = [np.linspace(-t_extent, t_extent, N, dtype=dtype)]
    coords.append(np.linspace(-extent, extent, N, dtype=dtype))
    coords.append(np.linspace(-extent, extent, N, dtype=dtype))
    if dim == 4:
        coords.append(np.linspace(-extent, extent, N, dtype=dtype))

    spacings = [float(c[1] - c[0]) for c in coords]

    if dim == 4:
        T, X, Y, Z = np.meshgrid(*coords, indexing="ij")
        shape = T.shape
    else:
        T, X, Y = np.meshgrid(*coords, indexing="ij")
        Z = None
        shape = T.shape

    def f_at_shift(shift):
        if dim == 4:
            rs2 = (X - v_s * (T + shift)) ** 2 + Y**2 + Z**2
        else:
            rs2 = (X - v_s * (T + shift)) ** 2 + Y**2
        rs = np.sqrt(rs2).astype(dtype, copy=False)
        return alcubierre_shape(rs, R, sigma).astype(dtype, copy=False)

    f = f_at_shift(0.0)
    f_minus = f_at_shift(-delta_tau)
    f_plus = f_at_shift(delta_tau)
    D2f = (f_plus - 2.0 * f + f_minus) / (delta_tau ** 2)
    S = D2f ** 2

    del f_minus, f_plus, D2f
    gc.collect()

    g = np.zeros((dim, dim) + shape, dtype=dtype)
    gi = np.zeros_like(g)

    g[0, 0] = -1.0 + v_s**2 * f**2
    g[0, 1] = -v_s * f
    g[1, 0] = -v_s * f
    g[1, 1] = 1.0
    g[2, 2] = 1.0
    if dim == 4:
        g[3, 3] = 1.0

    gi[0, 0] = -1.0
    gi[0, 1] = -v_s * f
    gi[1, 0] = -v_s * f
    gi[1, 1] = 1.0 - v_s**2 * f**2
    gi[2, 2] = 1.0
    if dim == 4:
        gi[3, 3] = 1.0

    dg = np.zeros((dim, dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            grads = np.gradient(g[a, b], *spacings, edge_order=2)
            for k in range(dim):
                dg[a, b, k] = grads[k]
            del grads

    Gamma = np.zeros((dim, dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                total = np.zeros(shape, dtype=dtype)
                for l in range(dim):
                    total += gi[a, l] * (dg[c, l, b] + dg[b, l, c] - dg[b, c, l])
                Gamma[a, b, c] = 0.5 * total
                del total

    del dg
    gc.collect()

    dGamma = np.zeros((dim, dim, dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                grads = np.gradient(Gamma[a, b, c], *spacings, edge_order=2)
                for k in range(dim):
                    dGamma[a, b, c, k] = grads[k]
                del grads

    Ricci = np.zeros((dim, dim) + shape, dtype=dtype)
    for b in range(dim):
        for c in range(dim):
            total = np.zeros(shape, dtype=dtype)
            for a in range(dim):
                total += dGamma[a, b, c, a] - dGamma[a, b, a, c]
                for d in range(dim):
                    total += Gamma[a, a, d] * Gamma[d, b, c]
                    total -= Gamma[a, c, d] * Gamma[d, b, a]
            Ricci[b, c] = total
            del total

    del dGamma
    gc.collect()

    R_scalar = np.zeros(shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            R_scalar += gi[a, b] * Ricci[a, b]

    Einstein = np.zeros((dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            Einstein[a, b] = Ricci[a, b] - 0.5 * g[a, b] * R_scalar

    Gmix = np.zeros_like(Einstein)
    for a in range(dim):
        for b in range(dim):
            for c in range(dim):
                Gmix[a, b] += gi[a, c] * Einstein[c, b]

    divG = divergence_mixed_from_geometry(Gmix, Gamma, spacings)

    dS = np.zeros((dim,) + shape, dtype=dtype)
    grads = np.gradient(S, *spacings, edge_order=2)
    for a in range(dim):
        dS[a] = grads[a]
    del grads

    ddS = np.zeros((dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        grads = np.gradient(dS[a], *spacings, edge_order=2)
        for b in range(dim):
            ddS[a, b] = grads[b]
        del grads

    Hess = np.zeros((dim, dim) + shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            Hess[a, b] = ddS[a, b]
            for l in range(dim):
                Hess[a, b] -= Gamma[l, a, b] * dS[l]

    del ddS
    gc.collect()

    BoxS = np.zeros(shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            BoxS += gi[a, b] * Hess[a, b]

    if dim == 4:
        dfdy = np.gradient(f, spacings[2], axis=2, edge_order=2)
        dfdz = np.gradient(f, spacings[3], axis=3, edge_order=2)
        rho_A = -(v_s**2) / (32.0 * np.pi) * (dfdy**2 + dfdz**2)
        del dfdy, dfdz
    else:
        dfdy = np.gradient(f, spacings[2], axis=2, edge_order=2)
        rho_A = -(v_s**2) / (32.0 * np.pi) * dfdy**2
        del dfdy

    nvec = np.zeros((dim,) + shape, dtype=dtype)
    nvec[0] = 1.0
    nvec[1] = v_s * f
    rho_num = np.zeros(shape, dtype=dtype)
    for a in range(dim):
        for b in range(dim):
            rho_num += Einstein[a, b] * nvec[a] * nvec[b]
    rho_num /= (8.0 * np.pi)
    del nvec

    return {
        "dim": dim, "N": N, "R": R, "sigma": sigma, "v_s": v_s,
        "extent": extent, "t_extent": t_extent, "delta_tau": delta_tau,
        "coords": coords, "spacings": spacings, "shape": shape,
        "f": f, "S": S, "g": g, "gi": gi, "Gamma": Gamma,
        "Ricci": Ricci, "R_scalar": R_scalar, "Einstein": Einstein,
        "Gmix": Gmix, "divG": divG, "dS": dS,
        "Hess": Hess, "BoxS": BoxS, "rho_A": rho_A, "rho_num": rho_num,
    }


In [4]:
# ============================================================
# 3. Stage 5 tensor metrics
# ============================================================

def make_Q_action_or_fit(geom, lam, beta):
    # Q = Hess(S) - g Box(S) - lambda g S + beta S G.
    dim = geom["dim"]
    g = geom["g"]
    S = geom["S"]
    Hess = geom["Hess"]
    BoxS = geom["BoxS"]
    Einstein = geom["Einstein"]

    Q = np.zeros((dim, dim) + geom["shape"], dtype=g.dtype)
    for a in range(dim):
        for b in range(dim):
            Q[a, b] = Hess[a, b] - g[a, b] * BoxS - lam * g[a, b] * S + beta * S * Einstein[a, b]
    return Q


def score_Q(geom, Qcov, crop=INTERIOR_CROP):
    Qmix = mix_tensor_up_down(Qcov, geom["gi"])
    C = divergence_mixed_from_geometry(Qmix, geom["Gamma"], geom["spacings"])
    Q_L2 = l2_components(Qmix, crop=crop, tensor_rank=2)
    C_L2 = l2_components(C, crop=crop, tensor_rank=1)
    return {
        "Qmix": Qmix, "C": C, "Q_L2": Q_L2, "C_L2": C_L2,
        "normalized_residual": C_L2 / Q_L2 if Q_L2 > 0 else np.nan,
    }


def analytic_fit_lambda_beta(geom, crop=INTERIOR_CROP):
    # Fit lambda and beta from residual = A - lambda B + beta D.
    dim = geom["dim"]
    dS_cov = geom["dS"]
    gi = geom["gi"]
    Ricci = geom["Ricci"]
    Einstein = geom["Einstein"]

    gradS_up = np.zeros_like(dS_cov)
    for a in range(dim):
        for b in range(dim):
            gradS_up[a] += gi[a, b] * dS_cov[b]

    A = np.zeros_like(dS_cov)
    D = np.zeros_like(dS_cov)
    B = dS_cov

    for nu in range(dim):
        for lam in range(dim):
            A[nu] += Ricci[nu, lam] * gradS_up[lam]
        for mu in range(dim):
            D[nu] += Einstein[mu, nu] * gradS_up[mu]

    interior = get_interior_slices(dS_cov.ndim - 1, crop)
    a = np.concatenate([A[nu][interior].ravel() for nu in range(dim)])
    b = np.concatenate([B[nu][interior].ravel() for nu in range(dim)])
    d = np.concatenate([D[nu][interior].ravel() for nu in range(dim)])

    M = np.vstack([-b, d]).T
    target = -a
    params, *_ = np.linalg.lstsq(M, target, rcond=None)
    lambda_fit, beta_fit = params

    def rms(v):
        return float(np.sqrt(np.mean(v**2)))

    residual_H = rms(a)
    residual_HT = rms(a - lambda_fit * b)
    residual_R = rms(a + beta_fit * d)
    residual_HTR = rms(a - lambda_fit * b + beta_fit * d)
    residual_action = rms(a - lambda_fit * b + BETA_ACTION * d)

    return {
        "lambda_fit": float(lambda_fit),
        "beta_fit": float(beta_fit),
        "analytic_residual_H": residual_H,
        "analytic_residual_HT": residual_HT,
        "analytic_residual_R": residual_R,
        "analytic_residual_HTR": residual_HTR,
        "analytic_residual_action_beta_minus_1": residual_action,
        "analytic_improvement_HTR_over_H": 1.0 - residual_HTR / residual_H if residual_H > 0 else np.nan,
        "analytic_action_over_fit": residual_action / residual_HTR if residual_HTR > 0 else np.nan,
    }


def integrate_scalar(geom, scalar, crop=INTERIOR_CROP):
    interior = get_interior_slices(scalar.ndim, crop)
    volume_element = np.prod(geom["spacings"])
    return float(np.sum(scalar[interior]) * volume_element)


def action_magnitude_metrics(geom, Q_action, Q_fit, crop=INTERIOR_CROP):
    rho = geom["rho_A"]
    Q00_action = Q_action[0, 0]
    Q00_fit = Q_fit[0, 0]
    interior = get_interior_slices(rho.ndim, crop)

    abs_rho_integral = integrate_scalar(geom, np.abs(rho), crop=crop)
    Q_action_pos_integral = integrate_scalar(geom, np.maximum(Q00_action, 0.0), crop=crop)
    Q_action_signed_integral = integrate_scalar(geom, Q00_action, crop=crop)
    Q_fit_pos_integral = integrate_scalar(geom, np.maximum(Q00_fit, 0.0), crop=crop)

    eta_needed_action_pos = abs_rho_integral / Q_action_pos_integral if Q_action_pos_integral > 0 else np.inf

    target = np.abs(rho[interior]).ravel()
    q = Q00_action[interior].ravel()
    q_fit = Q00_fit[interior].ravel()

    def corr(a, b):
        a = a - np.mean(a)
        b = b - np.mean(b)
        denom = np.sqrt(np.sum(a*a) * np.sum(b*b))
        return float(np.sum(a*b) / denom) if denom > 0 else np.nan

    return {
        "integral_abs_rho_A": abs_rho_integral,
        "integral_Q_action_positive": Q_action_pos_integral,
        "integral_Q_action_signed": Q_action_signed_integral,
        "integral_Q_fit_positive": Q_fit_pos_integral,
        "action_positive_fraction_of_abs_rho": Q_action_pos_integral / abs_rho_integral if abs_rho_integral > 0 else np.nan,
        "action_signed_fraction_of_abs_rho": Q_action_signed_integral / abs_rho_integral if abs_rho_integral > 0 else np.nan,
        "eta_needed_for_action_positive_to_match_abs_rho": eta_needed_action_pos,
        "action_Q00_absrho_correlation": corr(q, target),
        "fit_Q00_absrho_correlation": corr(q_fit, target),
    }


def relative_tensor_difference(geom, Q_action, Q_fit, crop=INTERIOR_CROP):
    Q_action_mix = mix_tensor_up_down(Q_action, geom["gi"])
    Q_fit_mix = mix_tensor_up_down(Q_fit, geom["gi"])
    D = Q_action_mix - Q_fit_mix
    D_L2 = l2_components(D, crop=crop, tensor_rank=2)
    F_L2 = l2_components(Q_fit_mix, crop=crop, tensor_rank=2)
    return D_L2 / F_L2 if F_L2 > 0 else np.nan


In [5]:
# ============================================================
# 4. Single-case Stage 5 evaluation
# ============================================================

def evaluate_case(dim, N, R, sigma, v_s, make_plots=False, label_prefix=""):
    t0 = time.time()
    memory_report(f"Before build R={R}, sigma={sigma}, v={v_s}:")

    geom = build_alcubierre_geometry(
        dim=dim, N=N, R=R, sigma=sigma, v_s=v_s,
        extent=EXTENT, t_extent=T_EXTENT, delta_tau=DELTA_TAU, dtype=DTYPE,
    )

    G_L2 = l2_components(geom["Gmix"], crop=INTERIOR_CROP, tensor_rank=2)
    divG_L2 = l2_components(geom["divG"], crop=INTERIOR_CROP, tensor_rank=1)
    relative_bianchi = divG_L2 / G_L2 if G_L2 > 0 else np.nan

    rho_error = geom["rho_num"] - geom["rho_A"]
    interior = get_interior_slices(rho_error.ndim, INTERIOR_CROP)
    rho_peak_error = float(np.max(np.abs(rho_error[interior])))
    rho_peak = float(np.max(np.abs(geom["rho_A"][interior])))
    rho_relative_peak_error = rho_peak_error / rho_peak if rho_peak > 0 else np.nan

    fit = analytic_fit_lambda_beta(geom, crop=INTERIOR_CROP)
    lambda_fit = fit["lambda_fit"]
    beta_fit = fit["beta_fit"]

    Q_fit = make_Q_action_or_fit(geom, lam=lambda_fit, beta=beta_fit)
    Q_action = make_Q_action_or_fit(geom, lam=lambda_fit, beta=BETA_ACTION)

    score_fit = score_Q(geom, Q_fit, crop=INTERIOR_CROP)
    score_action = score_Q(geom, Q_action, crop=INTERIOR_CROP)

    rel_diff = relative_tensor_difference(geom, Q_action, Q_fit, crop=INTERIOR_CROP)
    action_over_fit = score_action["normalized_residual"] / score_fit["normalized_residual"] if score_fit["normalized_residual"] > 0 else np.nan
    mag = action_magnitude_metrics(geom, Q_action, Q_fit, crop=INTERIOR_CROP)

    row = {
        "DIM": dim, "N": N, "R": R, "sigma": sigma, "v_s": v_s,
        "dt": geom["spacings"][0], "dx": geom["spacings"][1], "dy": geom["spacings"][2],
        "dz": geom["spacings"][3] if dim == 4 else np.nan,
        "relative_Bianchi_residual": relative_bianchi,
        "rho_relative_peak_error": rho_relative_peak_error,
        **fit,
        "beta_fit_minus_beta_action": beta_fit - BETA_ACTION,
        "abs_beta_fit_plus_1": abs(beta_fit + 1.0),
        "Q_fit_residual": score_fit["normalized_residual"],
        "Q_action_residual": score_action["normalized_residual"],
        "action_residual_over_fit_residual": action_over_fit,
        "relative_tensor_difference_action_vs_fit": rel_diff,
        **mag,
        "runtime_seconds": time.time() - t0,
        "ram_available_gib_after": memory_report().get("ram_available_gib", np.nan),
    }

    if make_plots:
        prefix = label_prefix or f"DIM{dim}_N{N}_R{R}_sigma{sigma}_v{v_s}".replace(".", "p")
        plot_central_scalar(geom["rho_A"], geom, f"rho_A {prefix}", "rho_A", f"{prefix}_rho_A.png")
        plot_central_scalar(Q_action[0,0], geom, f"Q_action_00 {prefix}", "Q_action_00", f"{prefix}_Q_action_00.png")
        plot_central_scalar(Q_action[0,0] - Q_fit[0,0], geom, f"Q_action_00 - Q_fit_00 {prefix}", "difference", f"{prefix}_Q_action_minus_fit_00.png")
        Cmag = np.sqrt(sum(score_action["C"][a]**2 for a in range(dim)))
        plot_central_scalar(Cmag, geom, f"Q_action residual {prefix}", "|nabla Q|", f"{prefix}_Q_action_residual.png")

    del geom, Q_fit, Q_action, score_fit, score_action
    gc.collect()
    return row


In [6]:
# ============================================================
# 5. Run Stage 5A parameter sweep
# ============================================================

results_path = OUTPUT_DIR / "stage5A_parameter_sweep_results.csv"
rows = []

cases = list(product(V_VALUES, SIGMA_VALUES, R_VALUES))
print(f"Running {len(cases)} sweep cases at DIM={DIM}, N={N_SWEEP}")
print(f"Approx scalar field size: {estimate_scalar_gib(N_SWEEP, DIM):.3f} GiB")

for i, (v_s, sigma, R) in enumerate(cases, start=1):
    print("\n" + "="*72)
    print(f"Case {i}/{len(cases)}: v_s={v_s}, sigma={sigma}, R={R}")
    try:
        row = evaluate_case(DIM, N_SWEEP, R=R, sigma=sigma, v_s=v_s, make_plots=False)
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)
        print("Completed:", {
            "beta_fit": row["beta_fit"],
            "action_over_fit": row["action_residual_over_fit_residual"],
            "rel_tensor_diff": row["relative_tensor_difference_action_vs_fit"],
            "Q_fraction": row["action_positive_fraction_of_abs_rho"],
            "runtime_s": row["runtime_seconds"],
        })
    except MemoryError as e:
        print("MEMORY ERROR:", e)
        rows.append({"DIM": DIM, "N": N_SWEEP, "R": R, "sigma": sigma, "v_s": v_s, "error": "MemoryError"})
        pd.DataFrame(rows).to_csv(results_path, index=False)
        gc.collect()
    except Exception as e:
        print("ERROR:", repr(e))
        rows.append({"DIM": DIM, "N": N_SWEEP, "R": R, "sigma": sigma, "v_s": v_s, "error": repr(e)})
        pd.DataFrame(rows).to_csv(results_path, index=False)
        gc.collect()

df_sweep = pd.DataFrame(rows)
df_sweep.to_csv(results_path, index=False)
display(df_sweep)


Running 48 sweep cases at DIM=4, N=31
Approx scalar field size: 0.007 GiB

Case 1/48: v_s=0.25, sigma=0.5, R=2.0
Before build R=2.0, sigma=0.5, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 14.894, 'ram_used_percent': 53.0}
Completed: {'beta_fit': -1.0013125671634129, 'action_over_fit': 0.9997989874161828, 'rel_tensor_diff': 2.3787368819598154e-05, 'Q_fraction': 1.008023856012504, 'runtime_s': 38.395904302597046}

Case 2/48: v_s=0.25, sigma=0.5, R=3.0
Before build R=3.0, sigma=0.5, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 15.041, 'ram_used_percent': 52.6}
Completed: {'beta_fit': -1.0032009640625057, 'action_over_fit': 0.9997758922590273, 'rel_tensor_diff': 3.696947980646425e-05, 'Q_fraction': 0.5655768074626857, 'runtime_s': 35.23930263519287}

Case 3/48: v_s=0.25, sigma=0.5, R=4.0
Before build R=4.0, sigma=0.5, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 15.148, 'ram_used_percent': 52.2}
Completed: {'beta_fit': -1.003166957350547, 'action_over_fit': 1

,DIM,N,R,sigma,v_s,dt,dx,dy,dz,relative_Bianchi_residual,...,integral_Q_action_positive,integral_Q_action_signed,integral_Q_fit_positive,action_positive_fraction_of_abs_rho,action_signed_fraction_of_abs_rho,eta_needed_for_action_positive_to_match_abs_rho,action_Q00_absrho_correlation,fit_Q00_absrho_correlation,runtime_seconds,ram_available_gib_after
0,4,31,2.0,0.5,0.25,0.026667,0.333333,0.333333,0.333333,0.002129,...,0.004486,-0.000558,0.004486,1.008024,-0.125406,0.992040,-0.137780,-0.137781,38.395904,13.339901
1,4,31,3.0,0.5,0.25,0.026667,0.333333,0.333333,0.333333,0.001631,...,0.003436,-0.000371,0.003436,0.565577,-0.061063,1.768106,-0.114808,-0.114808,35.239303,13.441154
2,4,31,4.0,0.5,0.25,0.026667,0.333333,0.333333,0.333333,0.001435,...,0.003606,0.000370,0.003606,0.513354,0.052698,1.947973,-0.177684,-0.177681,36.269133,13.638103
3,4,31,2.0,1.0,0.25,0.026667,0.333333,0.333333,0.333333,0.009408,...,0.032951,-0.000200,0.032951,6.331604,-0.038385,0.157938,-0.072434,-0.072435,30.997202,13.682545
4,4,31,3.0,1.0,0.25,0.026667,0.333333,0.333333,0.333333,0.009810,...,0.048849,-0.006884,0.048849,4.632247,-0.652781,0.215878,0.000693,0.000694,34.238418,13.711094
5,4,31,4.0,1.0,0.25,0.026667,0.333333,0.333333,0.333333,0.010147,...,0.067783,0.001445,0.067783,4.222399,0.090033,0.236832,0.000244,0.000244,35.238925,13.724606
6,4,31,2.0,2.0,0.25,0.026667,0.333333,0.333333,0.333333,0.042187,...,0.320154,0.000723,0.320176,36.615406,0.082648,0.027311,-0.045146,-0.045061,36.334604,13.628242
7,4,31,3.0,2.0,0.25,0.026667,0.333333,0.333333,0.333333,0.043775,...,0.628116,-0.009544,0.628170,32.096208,-0.487705,0.031156,-0.016592,-0.016533,40.800608,13.498177
8,4,31,4.0,2.0,0.25,0.026667,0.333333,0.333333,0.333333,0.045068,...,0.868086,-0.103124,0.868175,26.128501,-3.103919,0.038272,0.009854,0.009873,39.493376,13.567478
9,4,31,2.0,4.0,0.25,0.026667,0.333333,0.333333,0.333333,0.108240,...,2.995908,0.014589,2.996096,202.628105,0.986708,0.004935,-0.136736,-0.136671,39.942064,13.603859


In [7]:
# ============================================================
# 6. Optional selected high-resolution confirmation cases
# ============================================================

if RUN_CONFIRMATION:
    confirm_rows = []
    print(f"Running selected confirmation cases at DIM={DIM}, N={N_CONFIRM}")
    print(f"Approx scalar field size: {estimate_scalar_gib(N_CONFIRM, DIM):.3f} GiB")

    for i, case in enumerate(CONFIRMATION_CASES, start=1):
        print("\n" + "="*72)
        print(f"Confirmation {i}/{len(CONFIRMATION_CASES)}: {case}")
        try:
            prefix = f"confirm_DIM{DIM}_N{N_CONFIRM}_R{case['R']}_sigma{case['sigma']}_v{case['v_s']}".replace(".", "p")
            row = evaluate_case(DIM, N_CONFIRM, R=case["R"], sigma=case["sigma"], v_s=case["v_s"], make_plots=True, label_prefix=prefix)
            row["confirmation"] = True
            confirm_rows.append(row)
            pd.DataFrame(confirm_rows).to_csv(OUTPUT_DIR / "stage5B_confirmation_results.csv", index=False)
        except Exception as e:
            print("ERROR:", repr(e))
            confirm_rows.append({"DIM": DIM, "N": N_CONFIRM, **case, "confirmation": True, "error": repr(e)})
            pd.DataFrame(confirm_rows).to_csv(OUTPUT_DIR / "stage5B_confirmation_results.csv", index=False)
            gc.collect()

    df_confirm = pd.DataFrame(confirm_rows)
    display(df_confirm)
else:
    print("RUN_CONFIRMATION=False. Skipping high-resolution confirmation cases.")


Running selected confirmation cases at DIM=4, N=61
Approx scalar field size: 0.103 GiB

Confirmation 1/24: {'v_s': 0.25, 'sigma': 2.0, 'R': 2.0}
Before build R=2.0, sigma=2.0, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 16.191, 'ram_used_percent': 48.9}

Confirmation 2/24: {'v_s': 0.25, 'sigma': 2.0, 'R': 3.0}
Before build R=3.0, sigma=2.0, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 24.254, 'ram_used_percent': 23.5}

Confirmation 3/24: {'v_s': 0.25, 'sigma': 2.0, 'R': 4.0}
Before build R=4.0, sigma=2.0, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 22.379, 'ram_used_percent': 29.4}

Confirmation 4/24: {'v_s': 0.25, 'sigma': 4.0, 'R': 2.0}
Before build R=2.0, sigma=4.0, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 23.861, 'ram_used_percent': 24.8}

Confirmation 5/24: {'v_s': 0.25, 'sigma': 4.0, 'R': 3.0}
Before build R=3.0, sigma=4.0, v=0.25: {'ram_total_gib': 31.714, 'ram_available_gib': 22.308, 'ram_used_percent': 29.7}

Confirmation 6/24: {'v

,DIM,N,R,sigma,v_s,dt,dx,dy,dz,relative_Bianchi_residual,...,integral_Q_action_signed,integral_Q_fit_positive,action_positive_fraction_of_abs_rho,action_signed_fraction_of_abs_rho,eta_needed_for_action_positive_to_match_abs_rho,action_Q00_absrho_correlation,fit_Q00_absrho_correlation,runtime_seconds,ram_available_gib_after,confirmation
0,4,61,2.0,2.0,0.25,0.013333,0.166667,0.166667,0.166667,0.019955,...,0.001804,0.786322,77.154154,0.177055,0.012961,0.001227,0.001227,714.191353,7.616543,True
1,4,61,3.0,2.0,0.25,0.013333,0.166667,0.166667,0.166667,0.020523,...,0.002449,1.632622,71.952718,0.107938,0.013898,0.016403,0.016403,702.175233,6.000263,True
2,4,61,4.0,2.0,0.25,0.013333,0.166667,0.166667,0.166667,0.020861,...,-0.162305,2.651219,66.131381,-4.048491,0.015121,0.028950,0.028950,696.664984,5.904137,True
3,4,61,2.0,4.0,0.25,0.013333,0.166667,0.166667,0.166667,0.088822,...,-0.004873,9.492666,496.705497,-0.255025,0.002013,-0.006780,-0.006680,694.512651,7.332790,True
4,4,61,3.0,4.0,0.25,0.013333,0.166667,0.166667,0.166667,0.089784,...,-0.029252,20.536619,478.054273,-0.681008,0.002092,0.000207,0.000301,696.986083,6.033943,True
5,4,61,4.0,4.0,0.25,0.013333,0.166667,0.166667,0.166667,0.090141,...,-0.323462,35.716691,467.840718,-4.237365,0.002137,0.003202,0.003277,684.083852,6.225243,True
6,4,61,2.0,2.0,0.50,0.013333,0.166667,0.166667,0.166667,0.038732,...,0.121774,13.270838,325.517182,2.987136,0.003072,0.006855,0.006889,672.563520,7.814175,True
7,4,61,3.0,2.0,0.50,0.013333,0.166667,0.166667,0.166667,0.039771,...,0.178132,27.534832,303.361793,1.962652,0.003296,0.019986,0.020009,716.352540,6.704109,True
8,4,61,4.0,2.0,0.50,0.013333,0.166667,0.166667,0.166667,0.040412,...,-2.372617,44.839360,279.600752,-14.795773,0.003577,0.031207,0.031222,699.905075,6.821339,True
9,4,61,2.0,4.0,0.50,0.013333,0.166667,0.166667,0.166667,0.174182,...,-0.079943,162.142218,2119.143165,-1.045848,0.000472,0.001227,0.002055,684.422755,8.357815,True


In [8]:
# ============================================================
# 7. Stage 5 analysis plots
# ============================================================

results_path = OUTPUT_DIR / "stage5A_parameter_sweep_results.csv"
df_sweep = pd.read_csv(results_path)
if "error" in df_sweep.columns:
    df_ok = df_sweep[df_sweep["error"].isna()].copy()
else:
    df_ok = df_sweep.copy()

if len(df_ok) == 0:
    raise RuntimeError("No successful sweep cases to plot.")

summary = {
    "successful_cases": int(len(df_ok)),
    "failed_cases": int(len(df_sweep) - len(df_ok)),
    "N": int(df_ok["N"].iloc[0]),
    "DIM": int(df_ok["DIM"].iloc[0]),
    "beta_fit_mean": float(df_ok["beta_fit"].mean()),
    "beta_fit_std": float(df_ok["beta_fit"].std()),
    "abs_beta_fit_plus_1_median": float(df_ok["abs_beta_fit_plus_1"].median()),
    "action_over_fit_median": float(df_ok["action_residual_over_fit_residual"].median()),
    "relative_tensor_difference_median": float(df_ok["relative_tensor_difference_action_vs_fit"].median()),
    "action_fraction_median": float(df_ok["action_positive_fraction_of_abs_rho"].median()),
    "eta_needed_median": float(df_ok["eta_needed_for_action_positive_to_match_abs_rho"].median()),
}

with open(OUTPUT_DIR / "stage5_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "stage5_summary.csv", index=False)
print(json.dumps(summary, indent=2))

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["beta_fit"].mean().reset_index()
    plt.plot(sub["sigma"], sub["beta_fit"], marker="o", label=f"v={v}")
plt.axhline(-1.0, linestyle="--", label="beta_action=-1")
plt.xscale("log")
plt.title("Stage 5A: fitted curvature coupling versus wall sharpness")
plt.xlabel("sigma")
plt.ylabel("beta_fit")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_beta_fit_vs_sigma.png")

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["action_residual_over_fit_residual"].median().reset_index()
    plt.plot(sub["sigma"], sub["action_residual_over_fit_residual"], marker="o", label=f"v={v}")
plt.axhline(1.0, linestyle="--", label="action equals fit")
plt.xscale("log")
plt.title("Stage 5A: action residual relative to fitted residual")
plt.xlabel("sigma")
plt.ylabel("epsilon_action / epsilon_fit")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_action_over_fit_vs_sigma.png")

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["relative_tensor_difference_action_vs_fit"].median().reset_index()
    plt.plot(sub["sigma"], 100 * sub["relative_tensor_difference_action_vs_fit"], marker="o", label=f"v={v}")
plt.xscale("log")
plt.yscale("log")
plt.title("Stage 5A: action tensor difference from fitted tensor")
plt.xlabel("sigma")
plt.ylabel("relative tensor difference (%)")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_relative_tensor_difference_vs_sigma.png")

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["action_positive_fraction_of_abs_rho"].median().reset_index()
    plt.plot(sub["sigma"], sub["action_positive_fraction_of_abs_rho"], marker="o", label=f"v={v}")
plt.xscale("log")
plt.yscale("log")
plt.title("Stage 5A: action Q00 positive magnitude fraction")
plt.xlabel("sigma")
plt.ylabel("int max(Q_action_00,0) / int |rho_A|")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_action_magnitude_fraction_vs_sigma.png")

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["eta_needed_for_action_positive_to_match_abs_rho"].median().reset_index()
    plt.plot(sub["sigma"], sub["eta_needed_for_action_positive_to_match_abs_rho"], marker="o", label=f"v={v}")
plt.xscale("log")
plt.yscale("log")
plt.title("Stage 5A: eta needed for action Q00 to match |rho_A|")
plt.xlabel("sigma")
plt.ylabel("eta_needed")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_eta_needed_vs_sigma.png")

plt.figure(figsize=(9, 5))
for v in sorted(df_ok["v_s"].unique()):
    sub = df_ok[df_ok["v_s"] == v].groupby("sigma")["action_Q00_absrho_correlation"].median().reset_index()
    plt.plot(sub["sigma"], sub["action_Q00_absrho_correlation"], marker="o", label=f"v={v}")
plt.axhline(0.0, linestyle="--")
plt.xscale("log")
plt.title("Stage 5A: shape correlation between Q_action_00 and |rho_A|")
plt.xlabel("sigma")
plt.ylabel("correlation")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
savefig(OUTPUT_DIR / "stage5A_Q00_absrho_correlation_vs_sigma.png")

article_cols = [
    "DIM", "N", "R", "sigma", "v_s",
    "relative_Bianchi_residual", "rho_relative_peak_error",
    "lambda_fit", "beta_fit", "abs_beta_fit_plus_1",
    "Q_fit_residual", "Q_action_residual", "action_residual_over_fit_residual",
    "relative_tensor_difference_action_vs_fit",
    "action_positive_fraction_of_abs_rho",
    "eta_needed_for_action_positive_to_match_abs_rho",
    "action_Q00_absrho_correlation",
]
df_article = df_ok[article_cols].sort_values(["sigma", "v_s", "R"]).reset_index(drop=True)
df_article.to_csv(OUTPUT_DIR / "stage5_article_table.csv", index=False)
display(df_article)


{
  "successful_cases": 48,
  "failed_cases": 0,
  "N": 31,
  "DIM": 4,
  "beta_fit_mean": -0.705173577587466,
  "beta_fit_std": 0.33359577470056573,
  "abs_beta_fit_plus_1_median": 0.1845521953996655,
  "action_over_fit_median": 1.0001870644062365,
  "relative_tensor_difference_median": 0.0022662516440255503,
  "action_fraction_median": 80.55501685795608,
  "eta_needed_median": 0.0127496603426535
}


,DIM,N,R,sigma,v_s,relative_Bianchi_residual,rho_relative_peak_error,lambda_fit,beta_fit,abs_beta_fit_plus_1,Q_fit_residual,Q_action_residual,action_residual_over_fit_residual,relative_tensor_difference_action_vs_fit,action_positive_fraction_of_abs_rho,eta_needed_for_action_positive_to_match_abs_rho,action_Q00_absrho_correlation
0,4,31,2.0,0.5,0.25,0.002129,0.105399,0.004202,-1.001313,0.001313,0.001796,0.001796,0.999799,0.000024,1.008024,0.992040,-0.137780
1,4,31,3.0,0.5,0.25,0.001631,0.044585,0.003017,-1.003201,0.003201,0.002083,0.002083,0.999776,0.000037,0.565577,1.768106,-0.114808
2,4,31,4.0,0.5,0.25,0.001435,0.046053,0.003028,-1.003167,0.003167,0.002192,0.002192,1.000052,0.000023,0.513354,1.947973,-0.177684
3,4,31,2.0,0.5,0.50,0.004372,0.105511,0.016891,-1.004197,0.004197,0.007128,0.007128,0.999965,0.000164,4.368901,0.228890,-0.127975
4,4,31,3.0,0.5,0.50,0.003274,0.044580,0.012079,-1.011213,0.011213,0.008289,0.008291,1.000268,0.000276,2.434445,0.410771,-0.105131
5,4,31,4.0,0.5,0.50,0.002874,0.046051,0.012086,-1.011367,0.011367,0.008556,0.008562,1.000762,0.000178,2.185925,0.457472,-0.170339
6,4,31,2.0,0.5,0.75,0.006686,0.105750,0.038301,-1.005196,0.005196,0.015773,0.015774,1.000107,0.000329,11.167830,0.089543,-0.113828
7,4,31,3.0,0.5,0.75,0.004891,0.044572,0.027273,-1.018656,0.018656,0.018170,0.018184,1.000743,0.000733,6.164467,0.162220,-0.091609
8,4,31,4.0,0.5,0.75,0.004249,0.046047,0.027147,-1.019928,0.019928,0.018235,0.018262,1.001464,0.000498,5.443045,0.183721,-0.158257
9,4,31,2.0,0.5,1.00,0.008954,0.107817,0.068698,-0.998045,0.001955,0.026657,0.026655,0.999935,0.000174,23.460813,0.042624,-0.098721


In [9]:
# ============================================================
# 8. Create ZIP export package
# ============================================================

readme = f"""# Spinelli Stage 5 parameter sweep and magnitude test

This package contains the outputs of Spinelli_Stage5_parameter_sweep_and_magnitude_test.ipynb.

Configuration:
DIM = {DIM}
N_SWEEP = {N_SWEEP}
DTYPE = {DTYPE}
V_VALUES = {V_VALUES}
SIGMA_VALUES = {SIGMA_VALUES}
R_VALUES = {R_VALUES}
BETA_ACTION = {BETA_ACTION}

Core output files:
- stage5A_parameter_sweep_results.csv
- stage5_article_table.csv
- stage5_summary.csv
- stage5_summary.json

Core plots:
- stage5A_beta_fit_vs_sigma.png
- stage5A_action_over_fit_vs_sigma.png
- stage5A_relative_tensor_difference_vs_sigma.png
- stage5A_action_magnitude_fraction_vs_sigma.png
- stage5A_eta_needed_vs_sigma.png
- stage5A_Q00_absrho_correlation_vs_sigma.png

Interpretation targets:
- beta_fit close to -1 supports universality of the action-derived coupling.
- action_residual_over_fit_residual close to 1 means the beta=-1 action tensor performs nearly as well as the fitted HTR tensor.
- action_positive_fraction_of_abs_rho and eta_needed estimate whether the correction has physically meaningful magnitude.
"""

with open(OUTPUT_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme)

zip_path = Path(str(OUTPUT_DIR) + ".zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob("*"):
        zf.write(file_path, arcname=file_path.relative_to(OUTPUT_DIR))

print("Export complete")
print("Folder:", OUTPUT_DIR.resolve())
print("ZIP:", zip_path.resolve())


Export complete
Folder: C:\Users\spine\Documents\LaTeXProjects\Alcubierre test of the Spinelli Framework for discrete time\Stage 5 test\stage5_parameter_sweep_outputs
ZIP: C:\Users\spine\Documents\LaTeXProjects\Alcubierre test of the Spinelli Framework for discrete time\Stage 5 test\stage5_parameter_sweep_outputs.zip


## Notes on performance

For DIM=4:

- `N=31` is a practical pilot sweep.
- `N=41` is a stronger sweep but may take substantial time.
- `N=61` or `N=81` should be used only for selected confirmation cases.

A full 48-case sweep at `N=81` is likely not useful on a desktop unless the code is rewritten with chunked derivatives or moved to a high-memory workstation. The goal of this notebook is to establish whether the action-derived coupling \(eta=-1\) is robust across the parameter grid and whether the magnitude ratio is physically promising.
